In [2]:
import numpy as np, math
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar

import matplotlib.pyplot as plt
from scipy.optimize import brentq

In [3]:
# Constants
G=6.674e-11 
mE=5.972e24
mM=7.348e22
d=3.844e8

In [4]:
# System paramters and basic motion


w =math.sqrt(G*(mE+mM)/d**3) # angular velocity of the system

rE= mM*d/(mE+mM) # distance of earth from barycentre
rM= mE*d/(mE+mM) # distance of moon from barycentre


def body_pos(t): 
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    return ((-mM*d/(mE+mM)*cos_comp, -mM*d/(mE+mM)*sine_comp), (mE*d/(mE+mM)*cos_comp, mE*d/(mE+mM)*sine_comp))

def body_vel(t):
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    return ((mM*d/(mE+mM)*w*sine_comp, -mM*d/(mE+mM)*w*cos_comp), (-mE*d/(mE+mM)*w*sine_comp, mE*d/(mE+mM)*w*cos_comp))

def initial_conditions_planests(t0):                         # gives positon and velocity of each body at the start
    pos = body_pos(t0)
    vel = body_vel(t0)
    return (pos[0][0], pos[0][1], pos[1][0], pos[1][1],
            vel[0][0], vel[0][1], vel[1][0], vel[1][1])

def init(dr,t):                                                 #initial position of satellite dr past the moon at a time t 
    cos_comp ,sine_comp = np.cos(w*t), np.sin(w*t)
    return ((rM+dr)*cos_comp, (rM+dr)*sine_comp, 
            -w*(rM+dr)*sine_comp, w*(rM+dr)*cos_comp)

def func_to_solve(t, s):
    x, y, vx, vy = s             
    (xE, yE),(xM, yM) = body_pos(t)
    dEx,dEy,dMx,dMy=x-xE,y-yE,x-xM,y-yM
    dE3=(dEx*dEx+dEy*dEy)**1.5; dM3=(dMx*dMx+dMy*dMy)**1.5
    return [vx, vy , -G*mE*dEx/dE3 - G*mM*dMx/dM3, -G*mE*dEy/dE3 - G*mM*dMy/dM3]


In [24]:
#Scipy Solver 
def Scipy_solver(func,t_span, s, t_eval):
    sol = solve_ivp(func, t_span, s , t_eval=t_eval, method='RK45', rtol=1e-9, atol=1e-9)
    return sol 

#Scipy Solver with dense output  - FIX COS TS BROKEN
def Scipy_solver_dense(func, T0, T, n, dr):
    s = init(dr, T0)
    t = np.linspace(T0, T, n)
    sol = solve_ivp(func, [T0, T], s,t_eval=t,method='RK45',rtol=1e-9,atol=1e-9,dense_output=True)
    return sol

print(sol.y)
print(sol.t)

[[ 3.80727797e+08  3.80727794e+08  3.80727787e+08  3.80727775e+08
   3.80727757e+08  3.80727735e+08  3.80727708e+08  3.80727676e+08
   3.80727640e+08  3.80727598e+08  3.80727551e+08  3.80727500e+08
   3.80727443e+08  3.80727382e+08  3.80727316e+08  3.80727245e+08
   3.80727168e+08  3.80727087e+08  3.80727002e+08  3.80726911e+08
   3.80726815e+08  3.80726714e+08  3.80726609e+08  3.80726498e+08
   3.80726383e+08  3.80726262e+08  3.80726137e+08  3.80726007e+08
   3.80725872e+08  3.80725732e+08  3.80725587e+08  3.80725437e+08
   3.80725282e+08  3.80725123e+08  3.80724958e+08  3.80724788e+08
   3.80724614e+08  3.80724434e+08  3.80724250e+08  3.80724060e+08
   3.80723866e+08  3.80723667e+08  3.80723463e+08  3.80723253e+08
   3.80723039e+08  3.80722820e+08  3.80722596e+08  3.80722367e+08
   3.80722133e+08  3.80721894e+08  3.80721651e+08  3.80721402e+08
   3.80721148e+08  3.80720889e+08  3.80720625e+08  3.80720357e+08
   3.80720083e+08  3.80719804e+08  3.80719521e+08  3.80719232e+08
   3.80718

In [26]:
#Taylor Solver
def Taylor_solver(dr , T0, T,n):
    pos = np.array([init(dr, T0)])    
    dt= (T-T0)/n
    times = np.array([T0])
    for i in  range(0,n):
        t=T0+i*dt
        new_X = pos[-1][0] + dt*pos[-1][2] + 0.5*dt*dt*func_to_solve(times[-1],pos[-1])[2]
        new_y = pos[-1][1] + dt*pos[-1][3] + 0.5*dt*dt*func_to_solve(times[-1],pos[-1])[3]
        new_vx = pos[-1][2] + dt*func_to_solve(times[-1],pos[-1])[2]
        new_vy = pos[-1][3] + dt*func_to_solve(times[-1],pos[-1])[3]
        pos = np.append(pos, [[new_X, new_y, new_vx, new_vy]], axis=0)
        times = np.append(times, t+dt)
    return times, pos


In [ ]:
#Runga Kutta Solver


()


In [ ]:
#metrics for how accurate it is 

In [ ]:
#plots for each 